# Graph Neural Network (GNN) - TensorFlow / Keras

**Goal:** Classify nodes in a toy citation-style graph.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Nodes update by aggregating transformed neighbor features.
- **Where it is used:** social networks, molecules, recommendations, and citation graphs.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Graph Neural Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['nodes', 'neighbors', 'embeds']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.cos(x)*np.exp(-.1*x**2)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.25,.25,.30,.20])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['n0', 'n1', 'n2', 'n3'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
nodes, features, classes = 120, 16, 3
X = tf.random.normal((nodes, features))
labels = tf.cast(X[:, 0] + 0.5 * X[:, 1] > 0, tf.int32) + tf.cast(X[:, 2] > 1.0, tf.int32)
labels = tf.minimum(labels, classes - 1)
adjacency = tf.eye(nodes)
random_edges = tf.cast(tf.random.uniform((nodes, 5), maxval=nodes, dtype=tf.int32), tf.int32)
adjacency = tf.tensor_scatter_nd_update(
    adjacency,
    tf.reshape(tf.stack([tf.repeat(tf.range(nodes), 5), tf.reshape(random_edges, [-1])], axis=1), [-1, 2]),
    tf.ones(nodes * 5),
)
adjacency = tf.maximum(adjacency, tf.transpose(adjacency))
degree_inv_sqrt = tf.pow(tf.reduce_sum(adjacency, axis=1), -0.5)
norm_adj = degree_inv_sqrt[:, None] * adjacency * degree_inv_sqrt[None, :]
train_idx = tf.range(0, 80)
test_idx = tf.range(80, nodes)


In [ ]:
class GraphConvolution(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.dense = layers.Dense(units)

    def call(self, inputs):
        node_features, normalized_adjacency = inputs
        return tf.matmul(normalized_adjacency, self.dense(node_features))


class GCN(keras.Model):
    def __init__(self):
        super().__init__()
        self.gcn1 = GraphConvolution(32)
        self.gcn2 = GraphConvolution(classes)

    def call(self, inputs):
        x, normalized_adjacency = inputs
        x = tf.nn.relu(self.gcn1([x, normalized_adjacency]))
        return self.gcn2([x, normalized_adjacency])


model = GCN()
optimizer = keras.optimizers.AdamW(1e-2, weight_decay=5e-4)


In [ ]:
for epoch in range(120):
    with tf.GradientTape() as tape:
        logits = model([X, norm_adj])
        loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.gather(labels, train_idx),
            logits=tf.gather(logits, train_idx),
        ))
    optimizer.apply_gradients(zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables))
    if (epoch + 1) % 30 == 0:
        test_logits = tf.gather(model([X, norm_adj]), test_idx)
        accuracy = tf.reduce_mean(tf.cast(tf.argmax(test_logits, axis=1, output_type=tf.int32) == tf.gather(labels, test_idx), tf.float32))
        print(f"epoch={epoch+1:03d} test_accuracy={float(accuracy):.3f}")
